# Predicting arrears next month
Question: can we predict `arrears_next_month` from account information known at billing time?

Design decisions (following checklist.md):
- **Leakage:** `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time. It is dropped (it is almost a proxy for the target). `meter_ref` is an identifier and is dropped; `billing_month` is used only for splitting.
- **Features:** `tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`.
- **Test set:** the model is meant for future rows and accounts appear in many rows, so the test set is the latest months *and* whole accounts never seen in training.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from sklearn.base import clone
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
TARGET='arrears_next_month'
NUM=['occupants','kwh','bill_eur','late_payments_12m']; CAT=['tariff']; FEATURES=NUM+CAT
print(df.shape, df[TARGET].mean())

## 1. Test set (set aside before any modelling)
Test = 20% of accounts (whole accounts) restricted to the latest 6 months. Training = the other accounts, months before that cutoff. Rows of test accounts before the cutoff are unused.

In [ ]:
rng=np.random.RandomState(0)
meters=df.meter_ref.unique()
test_meters=set(rng.choice(meters, size=int(0.2*len(meters)), replace=False))
months=np.sort(df.billing_month.unique())
cutoff=months[-6]
is_test=df.meter_ref.isin(test_meters)&(df.billing_month>=cutoff)
is_train=~df.meter_ref.isin(test_meters)&(df.billing_month<cutoff)
train=df[is_train].reset_index(drop=True); test=df[is_test].reset_index(drop=True)
assert not set(train.meter_ref)&set(test.meter_ref) and train.billing_month.max()<test.billing_month.min()
print(len(train),len(test),train[TARGET].mean(),test[TARGET].mean())

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: standardised logistic regression, C in {0.1, 1, 10}.
3. `hgb`: histogram gradient boosting, max_depth in {2, 4}.

Primary metric: average precision (PR-AUC), since arrears is rare. Decision threshold chosen on CV to maximise F1.
Cross-validation (training rows only): expanding window by month, and entity-disjoint. Fold i validates on a block of later months for account group i only; it trains on earlier months of the other account groups. Preprocessing sits inside the pipeline so it is fit per fold.

In [ ]:
def make(model):
    pre=ColumnTransformer([('n',StandardScaler(),NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
    return Pipeline([('pre',pre),('m',model)])
candidates={'baseline':make(DummyClassifier(strategy='prior'))}
for C in [0.1,1,10]: candidates[f'logreg_C{C}']=make(LogisticRegression(C=C,max_iter=1000))
for d in [2,4]: candidates[f'hgb_d{d}']=make(HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=150,random_state=0))

tm=np.sort(train.billing_month.unique())
blocks=np.array_split(tm[len(tm)//2:],4)   # 4 validation blocks, later months
grp=pd.Series(train.meter_ref.unique()).sample(frac=1,random_state=1).reset_index(drop=True)
grp=pd.Series(np.arange(len(grp))%4,index=grp.values)   # 4 account groups
train_grp=train.meter_ref.map(grp).values
def folds():
    # fold i: validate on account group i in time block i; train on other accounts, strictly earlier months
    for i,b in enumerate(blocks):
        va=train.billing_month.isin(b).values&(train_grp==i)
        tr=(train.billing_month<b.min()).values&(train_grp!=i)
        yield np.where(tr)[0],np.where(va)[0]
def cv_oof(est):
    p=[];y=[]
    for tr,va in folds():
        m=clone(est).fit(train.loc[tr,FEATURES],train.loc[tr,TARGET])
        p.append(m.predict_proba(train.loc[va,FEATURES])[:,1]); y.append(train.loc[va,TARGET].values)
    return p,y
res={};oof={}
for n,e in candidates.items():
    p,y=cv_oof(e); oof[n]=(p,y)
    res[n]=dict(cv_ap=np.mean([average_precision_score(a,b) for a,b in zip(y,p)]),
                cv_auc=np.mean([roc_auc_score(a,b) for a,b in zip(y,p)]))
res=pd.DataFrame(res).T.sort_values('cv_ap',ascending=False); res

## 3. Choose model and threshold on CV only

In [ ]:
best=res.index[0]
P=np.concatenate(oof[best][0]); Y=np.concatenate(oof[best][1])
grid=np.linspace(0.02,0.9,45)
f1s=[f1_score(Y,P>=t) for t in grid]
THRESH=float(grid[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold:',round(THRESH,3),'| CV F1:',round(max(f1s),3))

**Choice (recorded before test scoring):** the model above, with features `occupants, kwh, bill_eur, late_payments_12m, tariff`, and the threshold above (maximising pooled CV F1). It is refit on all training rows and scored once on the test set below.

## 4. Final test score (once)

In [ ]:
final=clone(candidates[best]).fit(train[FEATURES],train[TARGET])
pt=final.predict_proba(test[FEATURES])[:,1]
base=clone(candidates['baseline']).fit(train[FEATURES],train[TARGET]).predict_proba(test[FEATURES])[:,1]
out=dict(test_ap=average_precision_score(test[TARGET],pt), test_auc=roc_auc_score(test[TARGET],pt),
         test_f1_at_threshold=f1_score(test[TARGET],pt>=THRESH),
         baseline_ap=average_precision_score(test[TARGET],base), prevalence=test[TARGET].mean())
pd.Series(out)

## Conclusion
See the test numbers above (headline = test average precision and AUC vs. the prevalence baseline). Caveats: results exclude `final_notice_sent`, which would look hugely predictive but is not available when the prediction is needed; the arrears rate rises sharply over time (about 2% in 2023-24 to about 25% by late 2025), so the CV-chosen threshold and absolute rates may not carry over to the test period; the test set contains a limited number of accounts and months, so uncertainty is nontrivial.